In [7]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display
import requests


In [4]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ


In [5]:
# Connect to client libraries

openai = OpenAI()
ollama_url = "http://localhost:11434/v1"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

ollama = OpenAI(api_key="ollama", base_url=ollama_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [21]:
models = ["gpt-5", "llama3.2", "gemini-3.5-flash-lite" ]

clients = {"gpt-5": openai, "llama3.2": ollama, "gemini-3.5-flash-lite": gemini}

pairs = ["BTCUSDT", "SOLUSDT", "BNBUSDT"]

markets = {"BTCUSDT": "bitcoin", "SOLUSDT": "solana", "BNBUSDT": "bnb"}

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [17]:

def get_market_data(tick):

    url = "https://api.binance.com/api/v3/ticker/24hr"

    params = {
        "symbol": tick
    }

    response = requests.get(url, params=params)

    data_market = response.json()

    return data_market

In [22]:
get_market_data("BTCUSDT")

{'symbol': 'BTCUSDT',
 'priceChange': '-237.72000000',
 'priceChangePercent': '-0.276',
 'weightedAvgPrice': '86299.21576215',
 'prevClosePrice': '86052.10000000',
 'lastPrice': '85814.38000000',
 'lastQty': '0.10343000',
 'bidPrice': '85814.38000000',
 'bidQty': '0.74448000',
 'askPrice': '85814.39000000',
 'askQty': '5.99866000',
 'openPrice': '86052.10000000',
 'highPrice': '87278.54000000',
 'lowPrice': '85461.53000000',
 'volume': '19728.27826000',
 'quoteVolume': '1702534942.17556190',
 'openTime': 1790070473012,
 'closeTime': 1790156873012,
 'firstId': 6703090660,
 'lastId': 6706191351,
 'count': 3100692}

In [25]:
def system_prompt_for(market):
    return f'''
    You task is predictor for {market}.
    Just give answer to buy or sell.
    Dont give explanation etc.
    '''

def user_prompt_for(market, data_market):
    return f"""
    Give me buy/sell signal for {market} market.
    I will send you the lastest price and other variable to make your prediction more clear.
    Dont explain about anything. Just Buy or Sell.

    This is lastest market data:
    {data_market}
    """


In [29]:
def message_for(market, data_market):
    return [
        {"role": "system", "content": system_prompt_for(market)},
        {"role": "user", "content": user_prompt_for(market, data_market)}
    ]

In [ ]:
def predict(market, data_market):
    response = 